In [121]:
#Importing libraries
import pandas as pd
import numpy as np
import yfinance as yf
import pandas_datareader.data as web
from datetime import datetime,timedelta
from dateutil import relativedelta

In [130]:
#Declare dates variables
# now
now = datetime.now()
# first day of the current month
f_day_of_month = now.replace(day=1)
#Declare the stock code, in this case is Apple
stock_code='MSFT'

In [131]:
#Download the close adjusted prices from the 1st day of the current month to the current day
df = yf.download(stock_code,
                 start=f_day_of_month.strftime('%Y-%m-%d'),
                 end=now.strftime('%Y-%m-%d'),
                 progress=False,
                 auto_adjust=False
                )

# Select the specific MultiIndex column for 'Adj Close' and the ticker 'AAPL',
# then convert it to a DataFrame with a single column named 'adj_close'.
df = df[('Adj Close',stock_code)].to_frame(name='adj_close')


In [132]:
# Calculate the simple and logarithmic returns
df['simple_rtn']= df.adj_close.pct_change()
df['log_rtn']= np.log(df.adj_close/df.adj_close.shift(1))
df

,adj_close,simple_rtn,log_rtn
Date,,,
2025-11-03,516.064148,NaN,NaN
2025-11-04,513.369202,-0.005222,-0.005236
2025-11-05,506.212555,-0.013941,-0.014039
2025-11-06,496.171356,-0.019836,-0.020035
2025-11-07,495.891876,-0.000563,-0.000563
2025-11-10,505.054718,0.018477,0.018309
2025-11-11,507.729706,0.005296,0.005282
2025-11-12,510.185150,0.004836,0.004824
2025-11-13,502.349792,-0.015358,-0.015477


## Returns with inflation rate

In [133]:

#Declare dates variables
from datetime import datetime
from dateutil.relativedelta import relativedelta
start_date = datetime.now().replace(month=1).replace(day=1)
end_date =  (((datetime.now()- relativedelta(months=1))).replace(day=1))- relativedelta(days=1)
start_date_2= (start_date - relativedelta(months=1)).replace(day=1)

#Download the close adjusted prices from the 1st day of the current month to the current day
df = yf.download(stock_code,
                 start=start_date.strftime('%Y-%m-%d'),
                 end=end_date.strftime('%Y-%m-%d'),
                 progress=False,
                 auto_adjust=False
                )

# Select the specific MultiIndex column for 'Adj Close' and the ticker 'AAPL',
# then convert it to a DataFrame with a single column named 'adj_close'.
df = df[('Adj Close',stock_code)].to_frame(name='adj_close')
#Create a dataframe with the merge between dates (left) and the close prices (right)
df_all_dates= pd.DataFrame(index=pd.date_range(start=start_date_2.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d')))
df=df_all_dates.join( df, how ='left') \
   .ffill() \
   .asfreq('ME')

# Download the data from CPI -inflation rate- US
df_cpi = web.DataReader('CPIAUCNS', 'fred', start=start_date_2.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'))
df_cpi.rename(columns={'CPIAUCNS': 'cpi'}, inplace=True)

# Align df_cpi index to month-end to match df's frequency
df_cpi_aligned = df_cpi.copy()
df_cpi_aligned.index = df_cpi_aligned.index + pd.offsets.MonthEnd(0)

# Resample to month-end frequency and forward-fill missing values (if any)
df_cpi_aligned = df_cpi_aligned.resample('ME').ffill()
df_cpi_aligned
#Merge between the inflation rate and the prices

# Merge df and the aligned df_cpi on their date indices
df_merged = pd.merge(df, df_cpi_aligned, left_index=True, right_index=True, how='left')

# Calculate the simple and inflation rate
df_merged ['simple_rtn']= df.adj_close.pct_change()
df_merged['inflation_rate']=df_merged.cpi.pct_change()

#Calculate the inflation-adjusted returns
df_merged['real_rtn']=(df_merged.simple_rtn + 1) / (df_merged.inflation_rate +1) - 1
df_merged

,adj_close,cpi,simple_rtn,inflation_rate,real_rtn
2024-12-31,NaN,315.605,NaN,NaN,NaN
2025-01-31,412.020630,317.671,NaN,0.006546,NaN
2025-02-28,394.873108,319.082,-0.041618,0.004442,-0.045856
2025-03-31,373.388306,319.799,-0.054409,0.002247,-0.056529
2025-04-30,393.152374,320.795,0.052932,0.003114,0.049663
2025-05-31,458.745819,321.465,0.166840,0.002089,0.164408
2025-06-30,495.665955,322.561,0.080481,0.003409,0.076809
2025-07-31,531.629395,323.048,0.072556,0.001510,0.070939
2025-08-31,505.743439,323.976,-0.048692,0.002873,-0.051417
2025-09-30,513.638611,324.800,0.015611,0.002543,0.013034
